In [1]:
import json
import traceback
from pathlib import Path

import aiida
from aiida import orm

from tqdm import tqdm

from mc3d_data_utils import verify_optimization_convergence, get_base_details_entry

AIIDA_PROFILE_NAME = "mc3d-pbe-v1"

aiida.load_profile(AIIDA_PROFILE_NAME)

LOG_DST = Path().resolve() / "log"

# delete old log:
if LOG_DST.exists():
    LOG_DST.unlink()

def log(msg):
    print(msg)
    with open(LOG_DST, 'a') as f:
        f.write(str(msg) + '\n')

In [2]:
# list aiida groups
! verdi -p {AIIDA_PROFILE_NAME} group list --all --all-users --count

  PK  Label                             Type string    User               Node count
----  --------------------------------  -------------  ---------------  ------------
   1  20250627-165843                   core.import    aiida@localhost        535005
   4  20250826-002539                   core.import    aiida@localhost         35240
   5  20250826-002710                   core.import    aiida@localhost         35240
  14  20250904-131108                   core.import    aiida@localhost         35240
   2  final_structures                  core           aiida@localhost         35240
  13  structures/final                  core           aiida@localhost         35240
  11  structures/final/converged        core           aiida@localhost         34970
  12  structures/final/converged_tight  core           aiida@localhost         33360
  10  structures/final/with_id          core           aiida@localhost         34970


In [3]:
# To understand data layout, generate node graph image for a random structure & associated output params.
struct_uuid = "8acb2196-425c-4540-a45c-99b9876189e8"
! verdi -p {AIIDA_PROFILE_NAME} node graph generate --process-out --process-in -f svg {struct_uuid} -O graph.svg

# we see that:
# * The final structure has no ancestors!
# * a PwBaseWorkchain originates from it that calls PwCalculations that produce the final output params.

/home/eimre_k/aiida/aiida-core/src/aiida/orm/nodes/data/code/legacy.py:602: AiidaDeprecationWarning: `Code.get_execname` method is deprecated, use `get_executable` instead. (this will be removed in v3)
  warn_deprecation('`Code.get_execname` method is deprecated, use `get_executable` instead.', version=3)
/home/eimre_k/aiida/aiida-core/src/aiida/orm/nodes/data/code/legacy.py:573: AiidaDeprecationWarning: `Code.is_local` method is deprecated, use a `PortableCode` instance and check the type. (this will be removed in v3)
  warn_deprecation(
/home/eimre_k/aiida/aiida-core/src/aiida/orm/nodes/data/code/legacy.py:524: AiidaDeprecationWarning: `Code.get_remote_exec_path` method is deprecated, use `InstalledCode.filepath_executable` instead. (this will be removed in v3)
  warn_deprecation(
Success: Output written to `graph.svg`


In [4]:
qb = (
    orm.QueryBuilder()
    .append(orm.Group, filters={"label": "structures/final/converged"}, tag="group")
    .append(orm.StructureData, with_group="group", project=["*"], tag="structure")
    .append(orm.WorkChainNode, with_incoming="structure", project=["*"], tag="wc")
    .append(
        orm.ArrayData,
        with_incoming="wc",
        edge_filters={"label": "output_trajectory"},
        project=["*"],
    )
    .append(
        orm.Dict,
        with_incoming="wc",
        edge_filters={"label": "output_parameters"},
        project=["*"],
        tag="parameters",
    )
    .append(orm.CalcJobNode, with_outgoing="parameters", project=["*"], tag="final_scf")
)

count = qb.count()
print(f"Number of structures: {count}")

Number of structures: 34970


In [11]:
qb.first()[3].attributes

/tmp/ipykernel_1430317/1879585454.py:1: AiidaDeprecationWarning: `Dict.attributes` is deprecated, use `Dict.base.attributes.all` instead. (this will be removed in v3)
  qb.first()[3].attributes


{'lsda': False,
 'energy': -53297.20551729,
 'volume': 487.30788008127,
 'fft_grid': [96, 80, 90],
 'energy_xc': -5538.2038633072,
 'wall_time': '   1m19.55s ',
 'rho_cutoff': 4353.821352096,
 'symmetries': [{'t_rev': '0', 'symmetry_number': 0},
  {'t_rev': '0', 'symmetry_number': 2},
  {'t_rev': '0', 'symmetry_number': 32},
  {'t_rev': '0', 'symmetry_number': 34}],
 'wfc_cutoff': 544.227669012,
 'exit_status': 0,
 'format_name': 'QEXSD',
 'lkpoint_dir': False,
 'occupations': 'smearing',
 'parser_info': 'aiida-quantumespresso parser pw.x v3.0.0a3',
 'total_force': 0.012058473637155,
 'creator_name': 'pwscf',
 'energy_ewald': -21473.802991929,
 'energy_units': 'eV',
 'fermi_energy': 7.6553,
 'forces_units': 'ev / angstrom',
 'q_real_space': False,
 'stress_units': 'GPascal',
 'charge_density': './charge-density.dat',
 'constraint_mag': 0,
 'energy_hartree': 28034.594794135,
 'format_version': '19.03',
 'parser_version': '3.0.0a3',
 'scf_iterations': 12,
 'beta_real_space': False,
 'cre

In [ ]:
data = {}

count_successful = 0
count_skipped = 0

for structure, workchain, opt_traj, out_params, final_scf in tqdm(
    qb.iterall(), total=count
):
    try:
        if verify_optimization_convergence(opt_traj) is None:
            log(f"SKIPPING: Thresholds not met for workchain {workchain}")
            count_skipped += 1
            continue
        
        extras = structure.base.extras.all
        mc3d_id = extras.get('mc3d_id')

        if not mc3d_id:
            log(f"SKIPPING: MC3D ID not assigned to {extras['source']}")
            count_skipped += 1
            continue

        details_entry = get_base_details_entry(structure, [out_params])
        details_entry["provenance_links"] = [
            {"label": "Final SCF calculation", "uuid": final_scf.uuid},
            {"label": "Final structure", "uuid": structure.uuid},
        ]
        data[mc3d_id] = details_entry

        count_successful += 1
    except Exception as exception:
        log(f"Found exception for {structure}: {exception}")
        traceback.print_exc()
        count_skipped += 1
        continue


log(f"Successful entries {count_successful}")
log(f"Skipped entries {count_skipped}")

OUTPUT_PATH = Path().resolve().parent

with (OUTPUT_PATH / "base.json").open("w") as handle:
    json.dump(data, handle, sort_keys=True, indent=2)

  0%|          | 0/34970 [00:00<?, ?it/s]/tmp/ipykernel_4077282/3231981968.py:15: AiidaDeprecationWarning: `StructureData.extras` is deprecated, use `StructureData.base.extras.all` instead. (this will be removed in v3)
  extras = structure.extras
/home/eimre_k/mcxd/discover-mc3d-data/common-library/mc3d_data_utils/__init__.py:38: AiidaDeprecationWarning: `StructureData.extras` is deprecated, use `StructureData.base.extras.all` instead. (this will be removed in v3)
  out_params = out_params_node.get_dict()
  3%|▎         | 1135/34970 [00:18<09:26, 59.73it/s]/home/eimre_k/micromamba/envs/aiida/lib/python3.10/site-packages/pymatgen/core/composition.py:1365: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
  9%|▉         | 3215/34970 [00:53<08:27, 62.57it/s]/home/eimre_k/micromamba/envs/aiida/lib/pytho

Successful entries 34970
Skipped entries 0
